## Conducción térmica:
$$\begin{array}{rll}\frac{\partial u}{\partial t} - \nabla\cdot (\kappa \nabla u) &= 0 & \text{en $\Omega\times (0,T)$,}\\
u(x,y,0) &= u_0 & \text{en }\Omega \\
u & = u_0 & \text{sobre }\Gamma_1\times (0,T) \\
\kappa \frac{\partial u}{\partial n} + \alpha(u-u_e) & = 0 & \text{sobre $\Gamma_2\times (0,T)$}\end{array}$$
con $\Omega=(0,L)\times (0,1)$, $\Gamma_1 = \{0,L\}\times(0,1)$, $\Gamma_2 = (0,L)\times \{0,1\}$, $L=6$

$u_e=25$, $\alpha=0.25$, $T=5$, $u_0(x,y)=10+\frac{90}{L}x$, y 
$\kappa(x,y) = \begin{cases} 2. & \text{ si }y<0.5\\ 0.2 & \text{ si }y>=0.5 \end{cases}$

$\Gamma_1$ tiene etiqueas asociadas 3,5; $\Gamma_2$ tiene etiquetas asociadas 2,4.

### Importamos módulos

In [ ]:
%reset -f
import mfem.ser as mfem
from glvis import glvis # visualización
import time # necesario para la visualización continua

### Lectura de malla

In [ ]:
mesh = mfem.Mesh.MakeCartesian2D(30,5,mfem.Geometry.SQUARE,False, 6.,1.)

In [ ]:
# Mostramos las etiquetas de la frontera que contiene la malla
print(mesh.bdr_attributes.ToList())
print(mesh.attributes.ToList())

#### Cambios en la malla

Cambiamos los atributos de la malla para poder definir el coeficiente $\kappa$ usando atributos. Buscamos el centro de cada elemento, y si la coordenada $y$ es menor que $0.5$ ponemos atributo = 2.

In [ ]:
for i in range(mesh.GetNE()):
    if mesh.GetElementCenterArray(i)[1]<0.5: # y<0.5
        mesh.SetAttribute(i,2)

mesh.SetAttributes()
print(mesh.attributes.ToList())

In [ ]:
mesh.Save("mallas/termico.mesh")

### Espacio de elementos finitos

In [ ]:
fec = mfem.H1_FECollection(1,  mesh.Dimension())
fespace = mfem.FiniteElementSpace(mesh, fec)
print('Número de incógnitas: ' +  str(fespace.GetTrueVSize()))

#### Etiquetas frontera

In [ ]:
# para las condiciones Robin
ess_list = [0]*mesh.bdr_attributes.Max()
ess_list[0] = 1
ess_list[2] = 1
ess_robin = mfem.intArray(ess_list)

# para la condiciones Dirichlet
ess_list = [0]*mesh.bdr_attributes.Max()
ess_list[1] = 1
ess_list[3] = 1
ess_dirich = mfem.intArray(ess_list)
print(ess_robin.ToList())
print(ess_dirich.ToList())

### Formulación variacional
Euler implícito para resolver el problema en tiempo
$$ \int_\Omega \left(\frac{u^n-u^{n-1} }{\delta t} w + \kappa \nabla u^n \nabla w\right) + \int_{\Gamma_2} \alpha(u^n-u_e)w = 0$$

que resulta
$$ \int_\Omega \left(\frac{u^n}{\delta t} w + \kappa \nabla u^n \nabla w\right) + \int_{\Gamma_2} \alpha u^n w =  \int_\Omega \frac{u^{n-1}}{\delta t}w + \int_{\Gamma_2} \alpha u_e w $$

#### Coeficientes

In [ ]:
alfa = 0.25
alfa_coeff = mfem.ConstantCoefficient(alfa)

ue_v = 25.
ue_coeff = mfem.ConstantCoefficient(ue_v*alfa)

class u0Coeff(mfem.PyCoefficient):
    def EvalValue(self,x):
        return 10.+90.*x[0]/6.
u0 = u0Coeff()    

kappa = mfem.PWConstCoefficient(mfem.Vector([0.2,2]))

# Coeficiente constante 1/dt
dt = 0.1
delta_t = mfem.ConstantCoefficient(1./dt)

# Coeficiente uold variable: representará u_{n-1} en la F.V. (dividido por dt)
uold = mfem.GridFunction(fespace)
uold_coef = mfem.ProductCoefficient(delta_t,mfem.GridFunctionCoefficient(uold))

# Coeficiente condición Robin (parte bilineal)
alpha = mfem.RestrictedCoefficient(alfa_coeff,ess_robin)

# Coeficiente condición Robin (parte lineal)
ue = mfem.RestrictedCoefficient(ue_coeff,ess_robin)

In [ ]:
# forma bilineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa))
a.AddDomainIntegrator(mfem.MassIntegrator(delta_t))
a.AddBoundaryIntegrator(mfem.MassIntegrator(alpha))
a.Assemble()

# forma lineal
b = mfem.LinearForm(fespace)
b.AddDomainIntegrator(mfem.DomainLFIntegrator(uold_coef))
b.AddBoundaryIntegrator(mfem.BoundaryLFIntegrator(ue))

#### Condición frontera

In [ ]:
# arrays para definir todas las etiquetas frontera
ess_tdof_list = mfem.intArray()

# recopilamos etiquetas para pasarlas al solver
fespace.GetEssentialTrueDofs(ess_dirich, ess_tdof_list)

In [ ]:
x = mfem.GridFunction(fespace)
x.ProjectBdrCoefficient(u0,ess_dirich)    

### Formulación del sistema

In [ ]:
A = mfem.SparseMatrix()
B = mfem.Vector()
X = mfem.Vector()
# Precondicionador tipo Gauss-Seidel
M = mfem.GSSmoother(A)

In [ ]:
# inicialización
uold.ProjectCoefficient(u0)

# Para la visualización, creamos la ventana con sus complementos
g = glvis((mesh, uold),400,400, keys="Rl*********")
g.render()
time.sleep(1) # necesario para que se actualice correctamente

t=0.
T = 5.

#### Bucle en tiempo
Solo hay que actualizar el segundo miembro

In [ ]:
    
while (t<T):    
    b.Update()
    b.Assemble()
    a.FormLinearSystem(ess_tdof_list, x, b, A, X, B)
    mfem.PCG(A, M, B, X, 0, 200, 1e-12, 0.0)
    a.RecoverFEMSolution(X, b, x)

    uold.ProjectGridFunction(x)

    g.update((mesh, x))
    time.sleep(.01)
    print(f"time: {t:.2f}", end="\r")
    t += dt